In [15]:
# positional encoding 

import torch 
import torch.nn as nn 
import torch.nn.functional as F


In [2]:
class PositionalEncoding(nn.Module):
    def __init__(self , max_length , emb_dim , dropout = 0.1):
        super().__init__()
        self.pos_emb = nn.Parameter(torch.randn(max_length , emb_dim) *0.02)
        self.dropout = nn.Dropout(dropout)
    def forward(self , x):
        x = x + self.pos_emb[:x.size(1)]
        return self.dropout(x)


In [3]:
class MultiheadAttention(nn.Module):
    def __init__(self , emb_dim , num_heads , dropout = 0.1):
        super().__init__()
        self.h = num_heads
        self.d = emb_dim // num_heads
        self.q_proj = nn.Linear(emb_dim , emb_dim)
        self.k_proj = nn.Linear(emb_dim , emb_dim)
        self.v_proj = nn.Linear(emb_dim , emb_dim)
        self.out_proj = nn.Linear(emb_dim , emb_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self , x):
        return x.view(x.size(0) , x.size(1) , self.h , self.d).transpose(1 , 2)
    def forward(self , query , key , value , attn_mask = None , key_padding_mask = None):
        q = self.split_heads(self.q_proj(query))
        k = self.split_heads(self.k_proj(key))
        v = self.split_heads(self.v_proj(value))
        scores = q @ k.transpose(2, 3) / (self.d ** 0.5)
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask , -torch.inf)
        if key_padding_mask is not None:
            mask = mask.masked_fill(key_padding_mask.unsqueeze(1).unsqueeze(2) )
            scores = scores.masked_fill(mask , -torch.inf)
        weights = F.softmax(scores , dim = -1)
        z= self.dropout(weights) @ v
        z = z.transpose(1 , 2)
        z= z.reshape(z.size(0) , z.size(1) , self.h * self.d)
        return (self.out_proj(z),weights)

      

In [4]:
class TransformerEncodingLayer(nn.Module):
    def __init__(self , d_model , num_heads , dim_feedforward=2048 , dropout = 0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model , num_heads , dropout)
        self.Leaner1 = nn.Linear(d_model , dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.Leaner2 = nn.Linear(dim_feedforward , d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self , src , src_mask = None , src_key_padding_mask = None):
        attn , _ = self.self_attn(src , src , src , attn_mask = src_mask , key_padding_mask = src_key_padding_mask)
        z= self.norm1(src + self.dropout(attn))
        ff = self.dropout(self.Leaner2(self.dropout(F.relu(self.Leaner1(z)))))
        z= self.norm2(z + ff)
        return z
        

In [5]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self , d_model , num_heads , dim_feedforward=2048 , dropout = 0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model , num_heads , dropout)
        self.multihead_attn = MultiheadAttention(d_model , num_heads , dropout)
        self.Leaner1 = nn.Linear(d_model , dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.Leaner2 = nn.Linear(dim_feedforward , d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self , tgt , memory , tgt_mask = None , memory_mask = None ,
                tgt_key_padding_mask = None , memory_key_padding_mask = None):
        attn1 , _ = self.self_attn(tgt , tgt , tgt , attn_mask = tgt_mask ,
                                   key_padding_mask = tgt_key_padding_mask)
        z= self.norm1(tgt + self.dropout(attn1))
        attn2 , _ = self.multihead_attn(z , memory , memory ,
                                        attn_mask = memory_mask ,
                                        key_padding_mask = memory_key_padding_mask)
        z= self.norm2(z + self.dropout(attn2))
        ff = self.dropout(self.Leaner2(self.dropout(F.relu(self.Leaner1(z)))))
        z= self.norm3(z + ff)
        return z

In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader
from datasets import load_dataset
import tokenizers

In [25]:
nmt_original_valid_set , nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name= "eng-spa",
    split=["validation" , "test"])

split = nmt_original_valid_set.train_test_split(train_size = 0.8 ,seed=42)
nmt_train_set , nmt_valid_set = split["train"] ,split["test"]

In [26]:
def train_eng_spa():
    for pair in nmt_train_set:
        yield pair["source_text"]
        yield pair["target_text"]

max_length = 256
vocab_size = 10_000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id = 0 , pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer  =tokenizers.trainers.BpeTrainer(vocab_size=vocab_size , special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa() , nmt_tokenizer_trainer)


In [27]:
from collections import namedtuple
fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [29]:
def nmt_collate_fn(batch):
    src_texts = [pair["source_text"] for pair in batch]
    tgt_texts = [f"<s> {pair['target_text']} </s>" for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids ,src_mask ,tgt_token_ids[: ,:-1] , tgt_mask[:, :-1])
    labels = tgt_token_ids[: ,1:]
    return inputs , labels

In [32]:
batch_size =32

nmt_train_loader = DataLoader(nmt_train_set , batch_size=batch_size , collate_fn = nmt_collate_fn ,shuffle = True)
nmt_valid_loader = DataLoader(nmt_valid_set , batch_size=batch_size , collate_fn = nmt_collate_fn )
nmt_test_loader = DataLoader(nmt_test_set , batch_size=batch_size , collate_fn = nmt_collate_fn )

In [35]:
class NmtTransformer(nn.Module):
    def __init__(self , vocab_size , max_length , embed_dim =512 , pad_id = 0 , num_heads = 8 , num_layers = 6 , dropout = 0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size , embed_dim , padding_idx = pad_id)
        self.pos_embed = PositionalEncoding(max_length , embed_dim , dropout)
        self.transformer = nn.Transformer(embed_dim , nhead = num_heads , num_encoder_layers = num_layers ,
                                          num_decoder_layers = num_layers , batch_first = True , dropout = dropout)
        self.output =  nn.Linear(embed_dim , vocab_size)
    def forward(self , pair):
        src_embed = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embed = self.pos_embed(self.embed(pair.tgt_token_ids))
        src_pad_mask = ~pair.src_mask.bool()
        tgt_pad_mask = ~pair.tgt_mask.bool()
        size = [pair.tgt_token_ids.size(1)] *2
        full_mask = torch.full(size , True , device=tgt_pad_mask.device )
        causal_mask = torch.triu(full_mask , diagonal=1)
        out_decoder = self.transformer(src_embed , tgt_embed , src_key_padding_mask = src_pad_mask , memory_key_padding_mask = src_pad_mask , 
                                       tgt_mask = causal_mask , tgt_is_causal = True , tgt_key_padding_mask = tgt_pad_mask)
        return self.output(out_decoder).permute(0 ,2, 1)


torch.manual_seed(42)
vocab_size = nmt_tokenizer.get_vocab_size()
nmt_tr_model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0,
                            num_heads=4, num_layers=2, dropout=0.1)

In [36]:
import torch
import torch.nn as nn
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────────────────
# 1. Device, Loss Function & Optimizer
# ─────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

nmt_tr_model.to(device)

# CrossEntropyLoss expects:
#   Inputs:  (batch_size, num_classes, seq_len)
#   Targets: (batch_size, seq_len)
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(nmt_tr_model.parameters(), lr=1e-3)

# ─────────────────────────────────────────────────────────────
# 2. Training Loop
# ─────────────────────────────────────────────────────────────
num_epochs = 10

for epoch in range(1, num_epochs + 1):
    # ── Training Phase ──
    nmt_tr_model.train()
    train_loss = 0.0
    correct_tokens = 0
    total_tokens = 0

    train_pbar = tqdm(nmt_train_loader, desc=f"Epoch {epoch:02d}/{num_epochs} [Train]")
    for X_batch, y_batch in train_pbar:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        # 1. Forward pass
        logits = nmt_tr_model(X_batch)  # shape: (batch, vocab_size=39, seq_len)

        # 2. Compute loss
        loss = loss_fn(logits, y_batch)

        # 3. Backward pass & optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        # Compute accuracy across characters
        # Argmax along dimension 1 (vocab_size)
        preds = logits.argmax(dim=1)  # shape: (batch, seq_len)
        correct_tokens += (preds == y_batch).sum().item()
        total_tokens += y_batch.numel()

        train_pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{(correct_tokens / total_tokens):.3f}"
        })

    avg_train_loss = train_loss / len(nmt_train_loader)
    train_acc = correct_tokens / total_tokens

    # ── Validation Phase ──
    nmt_tr_model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for X_batch, y_batch in nmt_valid_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = nmt_tr_model(X_batch)
            loss = loss_fn(logits, y_batch)

            val_loss += loss.item()
            preds = logits.argmax(dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.numel()

    avg_val_loss = val_loss / len(nmt_valid_loader)
    val_acc = val_correct / val_total

    print(f"\nEpoch {epoch:02d} Summary: "
          f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.3f} || "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.3f}\n")



Using device: cpu


Epoch 01/10 [Train]:  21%|██        | 1020/4933 [06:42<25:44,  2.53it/s, loss=2.2505, acc=0.733]  


KeyboardInterrupt: 

In [38]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([{"source_text": src_text,
                                    "target_text": tgt_text}])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID
            next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [39]:
nmt_tr_model.eval()
translate(nmt_tr_model,"I like to play soccer with my friends at the beach")

c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(


' Me gusta la puerta a mi padre . </s>'

In [2]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [ ]:
from transformers import BertConfig , BertForMaskedLM , BertTokenizerFast 

bert_tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
config = BertConfig(
    vocab_size=bert_tokenizer.vocab_size,
    hidden_size=128 , num_hidden_layers=2 , num_attention_heads=4 , intermediate_size=512 , max_position_embeddings=512 
)
bert_model = BertForMaskedLM(config)

c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from datasets import load_dataset 

def tokenize(examples , tokenizer= bert_tokenizer):
    return bert_tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

mlm_dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
mlm_dataset = mlm_dataset.map(tokenize, batched=True)
 

In [5]:
from transformers import Trainer , TrainingArguments
from transformers import DataCollatorForLanguageModeling

arg = TrainingArguments(output_dir = './myBert' , num_train_epochs=5 , per_device_train_batch_size=16 )

mlm_collator = DataCollatorForLanguageModeling(bert_tokenizer, mlm_probability=0.15 ,mlm=True)
trainer = Trainer(model=bert_model , args=arg , train_dataset=mlm_dataset , data_collator=mlm_collator)
trainer_output = trainer.train()


Step,Training Loss
500,8.906749
1000,7.506526
1500,7.330773
2000,7.232852
2500,7.171374
3000,7.130139
3500,7.122299
4000,7.062300
4500,7.071129
5000,7.010335


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 36.11it/s]


In [6]:
from transformers import pipeline

torch.manual_seed(42)
fill_mask = pipeline("fill-mask" , model=bert_model , tokenizer=bert_tokenizer)
top_predictions = fill_mask("The capital of [MASK] is Rome.")
top_predictions[0]


{'score': 0.05051824450492859,
 'token': 1996,
 'token_str': 'the',
 'sequence': 'the capital of the is rome.'}

In [7]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = ["She's shopping", "She bought some shoes", "She's working"]
embeddings = model.encode(sentences ,convert_to_tensor=True)
similarites = model.similarity(embeddings , embeddings)




c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELGRM\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\hu

In [1]:
from transformers import AutoTokenizer , AutoModelForCausalLM
model_id = 'gpt2'
gpt_tokenizer = AutoTokenizer.from_pretrained(model_id)
gpt2 = AutoModelForCausalLM.from_pretrained(model_id , device_map="auto" , dtype='auto')



c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ELGRM\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\ELGRM\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:149: UserWarning:

In [11]:
def generate(model , tokenizer ,prompt, max_new_tokens=50 , **generate_kwargs):
    inputs = tokenizer(prompt , return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                pad_token_id=tokenizer.eos_token_id,
                                 **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)



In [17]:
torch.manual_seed(42)
prompt = "Scientists found a talking unicorn today. Here's the full story:"

generate(gpt2, gpt_tokenizer, prompt, do_sample=True , top_p=0.6)

"Scientists found a talking unicorn today. Here's the full story:\n\nThe creature is said to have been a member of the species known as the Unicorn, which has been found on the island of Oceti in southern Africa.\n\nThe unicorn, a type of unicorn found in many other species,"

In [22]:
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

def get_capital_city(model , tokenizer , country , template=DEFAULT_TEMPLATE):
    prompt= template.format(country=country)
    text = generate(model , tokenizer , prompt , max_new_tokens=10)
    answer = text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

print(get_capital_city(gpt2, gpt_tokenizer, "United Kingdom"))   
print(get_capital_city(gpt2, gpt_tokenizer, "Mexico"))


London
Mexico City
